# 🐍 Python for AI Development — 核心必备基础

本 Notebook 覆盖构建 AI 应用（尤其是 LangChain/LangGraph/Agent 系统）所必需的 Python 高级特性。

**学习目标：**
- 掌握 async/await 异步编程，为生产级 LLM API 调用打基础
- 理解 TypedDict / Pydantic / Dataclass 三种结构化数据方式的区别与选型
- 熟练使用 Type Hints 提升代码可读性与 IDE 支持
- 掌握上下文管理器（Context Manager）用于资源管理
- 将以上所有概念组合成一个完整的异步文档处理函数

**先决条件：** Python 3.10+ 基础语法

---
## 1. async/await 基础

### 为什么要学？
所有生产级 LLM API 调用都应该是异步的。当你同时调用多个 LLM（比如并行评估、批量处理文档），
同步代码会让 CPU 大量时间浪费在等待网络 I/O 上。async/await 让一个线程可以高效地切换协程，
在等待一个 API 响应时去处理另一个请求。

In [ ]:
"""
async/await 基础：从同步到异步
=================================
关键概念：
- async def: 定义一个协程（coroutine）
- await: 暂停当前协程，等待另一个协程完成
- asyncio.gather(): 并发运行多个协程
- asyncio.run(): 顶层入口，运行协程
"""
import asyncio
import time
from typing import List

# ============================================================
# 示例 1: 同步 vs 异步 —— 直观感受性能差异
# ============================================================

def sync_fetch(url: str, delay: float) -> str:
    """模拟同步 API 调用 —— 阻塞等待"""
    time.sleep(delay)  # 模拟网络 I/O
    return f"[同步] 从 {url} 获取数据 (耗时 {delay}s)"

async def async_fetch(url: str, delay: float) -> str:
    """模拟异步 API 调用 —— 非阻塞等待"""
    await asyncio.sleep(delay)  # 模拟网络 I/O，释放控制权
    return f"[异步] 从 {url} 获取数据 (耗时 {delay}s)"

# 测试：3 个请求，每个模拟 1 秒延迟
urls = [("api/v1", 1.0), ("api/v2", 1.0), ("api/v3", 1.0)]

print("=" * 60)
print("同步版本：")start = time.perf_counter()
sync_results = [sync_fetch(url, delay) for url, delay in urls]
print(f"  总耗时: {time.perf_counter() - start:.2f}s")
for r in sync_results:
    print(f"  {r}")

print()
print("异步版本：")async def run_async():
    """并发运行三个异步任务"""
    tasks = [async_fetch(url, delay) for url, delay in urls]
    return await asyncio.gather(*tasks)

start = time.perf_counter()
async_results = asyncio.run(run_async())
print(f"  总耗时: {time.perf_counter() - start:.2f}s")
for r in async_results:
    print(f"  {r}")

print()
print(f"✅ 加速比: 同步 ~3s vs 异步 ~1s（约 3x 提升）")

In [ ]:
# ============================================================
# 示例 2: 异步函数中的错误处理
# ============================================================

async def fetch_with_error(url: str, should_fail: bool = False) -> str:
    """
    模拟可能失败的 API 调用。
    
    生产建议：
    - 总是用 try/except 包裹 await 调用
    - 区分可重试错误（超时、限流）和不可重试错误（认证失败）
    - 使用 asyncio.wait_for() 设置超时
    """
    await asyncio.sleep(0.5)
    if should_fail:
        raise ConnectionError(f"无法连接到 {url}")
    return f"✅ 成功: {url}"

async def robust_fetch(urls_with_flags: List[tuple]) -> dict:
    """
    健壮的批量异步请求：
    - 单个失败不影响其他
    - 返回 (成功列表, 失败列表) 分离的结果
    
    何时使用：
    - 批量调用多个 LLM API 端点
    - 并行检索多个文档源
    - 同时评估多个候选回答
    """
    async def safe_fetch(url: str, should_fail: bool) -> tuple:
        """包装单个请求，捕获异常并返回元组 (url, result_or_error, success)"""
        try:
            # 设置 2 秒超时（生产环境常用模式）
            result = await asyncio.wait_for(
                fetch_with_error(url, should_fail),
                timeout=2.0
            )
            return (url, result, True)
        except asyncio.TimeoutError:
            return (url, "⏰ 请求超时", False)
        except ConnectionError as e:
            return (url, f"🔌 {e}", False)
        except Exception as e:
            return (url, f"❌ 未知错误: {e}", False)
    
    tasks = [safe_fetch(url, fail) for url, fail in urls_with_flags]
    results = await asyncio.gather(*tasks)
    
    return {
        "successes": [r for r in results if r[2]],
        "failures": [r for r in results if not r[2]]
    }

# 测试：3 个请求，其中 1 个会失败
test_urls = [
    ("https://api.openai.com/v1/chat", False),
    ("https://api.anthropic.com/v1/messages", True),   # 模拟失败
    ("https://api.cohere.com/v1/generate", False),
]

result = asyncio.run(robust_fetch(test_urls))
print(f"✅ 成功 {len(result['successes'])} 个:")
for url, msg, _ in result['successes']:
    print(f"    {msg}")
print(f"❌ 失败 {len(result['failures'])} 个:")
for url, msg, _ in result['failures']:
    print(f"    {msg}")

In [ ]:
# ============================================================
# 示例 3: asyncio.gather 的 return_exceptions 参数
# ============================================================

async def demo_return_exceptions():
    """
    使用 return_exceptions=True 时，
    协程抛出的异常不会被传播，而是作为返回值返回。
    适用场景：你希望所有任务都完成，然后再统一处理错误。
    """
    async def may_fail(n: int) -> float:
        await asyncio.sleep(0.1 * n)
        if n == 2:
            raise ValueError(f"任务 {n} 失败")
        return 100 / n
    
    results = await asyncio.gather(
        may_fail(1), may_fail(2), may_fail(4),
        return_exceptions=True  # 关键：异常变成返回值
    )
    
    for i, r in enumerate(results, 1):
        if isinstance(r, Exception):
            print(f"  任务 {i}: 异常 → {type(r).__name__}: {r}")
        else:
            print(f"  任务 {i}: 结果 → {r}")

asyncio.run(demo_return_exceptions())
print()
print("💡 提示: return_exceptions=True 常用于需要部分成功场景")

### ✏️ 练习 1: async/await

1. 编写一个 `async def batch_llm_call(prompts: list) -> list` 函数，并发处理 5 个 prompt，每个模拟 0.2s-0.8s 随机延迟，返回结果列表。
2. 添加超时机制：如果单个调用超过 0.5s，则返回 "TIMEOUT" 而不是抛异常。
3. 使用 `asyncio.as_completed()` 逐结果打印进度（而非等全部完成）。

---
## 2. TypedDict —— 类型安全字典

### 为什么要学？
LangGraph 的 State 就是 TypedDict。了解 TypedDict 是理解 LangGraph 状态管理的基础。
它让你在保持字典灵活性的同时获得类型检查和 IDE 自动补全。

In [ ]:
"""
TypedDict 详解
===============
- 运行时仍是普通 dict（零性能开销）
- 仅用于静态类型检查（mypy / pyright）
- 支持 Optional、Required/NotRequired（Python 3.11+）
- LangGraph State 就是 TypedDict 的子类
"""
from typing import TypedDict, Optional, List, Dict, Any

# 在 Python 3.11+ 中可从 typing 导入 NotRequired, Required
# 在 Python 3.10 中可用 typing_extensions
try:
    from typing import NotRequired, Required  # Python 3.11+
except ImportError:
    from typing_extensions import NotRequired, Required  # Python 3.10

print("✅ TypedDict 导入成功")

# ============================================================
# 示例 1: 基本 TypedDict 定义与使用
# ============================================================

class ChatMessage(TypedDict):
    """
    聊天消息的 TypedDict 定义。
    
    为什么用 TypedDict？
    - 不需要 Pydantic 的验证开销
    - 与 JSON 序列化天然兼容
    - 作为 LangGraph State 的基础类型
    """
    role: str                          # 必填字段
    content: str                       # 必填字段
    name: NotRequired[str]             # Python 3.11+: 可选字段（不传不报错）
    metadata: NotRequired[Dict[str, Any]]  # 可选元数据

# 使用：IDE 会有自动补全！
msg: ChatMessage = {
    "role": "user",
    "content": "什么是 LangGraph？",
}

print(f"消息角色: {msg['role']}")       # IDE 补全: msg['role']
print(f"消息内容: {msg['content']}")    # IDE 补全: msg['content']
print(f"类型: {type(msg)}")             # 运行时仍是 <class 'dict'>！
print(f"是 dict? {isinstance(msg, dict)}")  # True

In [ ]:
# ============================================================
# 示例 2: TypedDict vs 普通 dict 对比
# ============================================================

class AgentState(TypedDict):
    """
    LangGraph 风格的 Agent 状态定义。
    每个字段代表图中一个可观测的状态通道。
    """
    messages: List[ChatMessage]        # 对话历史
    next_step: str                     # 下一步动作
    tool_results: NotRequired[List[str]]  # 工具调用结果（可选）
    error: NotRequired[Optional[str]]  # 错误信息（可选，可为 None）

# === TypedDict 的好处 ===
# 1. 类型检查器会捕获错误
try:
    state: AgentState = {
        "messages": [],
        # 缺少 'next_step' → mypy 会报错！
    }
except Exception:
    pass  # 运行时不会有问题，但静态检查会报错

# 2. IDE 自动补全
correct_state: AgentState = {
    "messages": [],
    "next_step": "agent",
}
# 输入 correct_state['  → IDE 会提示 messages, next_step, tool_results, error

print("✅ AgentState 定义完成")
print(f"  必填字段: messages, next_step")
print(f"  可选字段: tool_results, error")

# 3. 函数签名获得类型安全
def process_state(state: AgentState) -> AgentState:
    """处理 Agent 状态并返回更新后的状态（LangGraph 节点模式）"""
    return {
        "messages": state["messages"],
        "next_step": "tools" if state.get("tool_results") else "end",
    }

print(f"  处理后 next_step: {process_state(correct_state)['next_step']}")

### ✏️ 练习 2: TypedDict

1. 定义一个 `RAGState(TypedDict)`，包含：`query`, `retrieved_docs`, `answer`, `sources`（可选）。
2. 编写函数 `def retrieve(state: RAGState) -> RAGState`，模拟从 state 中取 query 进行检索。
3. 在 mypy/pyright 检查下验证：缺少必填字段时会报错。

---
## 3. Pydantic BaseModel 深度剖析

### 为什么要学？
Pydantic 是整个课程中结构化输出（Structured Output）的核心工具。
LLM 返回的 JSON 需要用 Pydantic 验证和解析，配置文件用 Pydantic Settings 管理，
工具参数的定义也离不开 Pydantic。

In [ ]:
"""
Pydantic BaseModel 深度剖析
============================
核心概念：
- BaseModel: 数据模型的基类
- Field(): 字段级别的验证和元数据
- @field_validator: 自定义字段验证逻辑
- model_dump(): 导出为字典（序列化）
- model_validate(): 从字典创建模型（反序列化）
"""
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional, Literal, List
from datetime import datetime
from enum import Enum

print("✅ Pydantic 导入成功")

# ============================================================
# 示例 1: 基本 BaseModel —— LLM 结构化输出
# ============================================================

class SentimentEnum(str, Enum):
    """情感标签枚举 —— 限定 LLM 只能输出这几种值"""
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"

class SentimentResult(BaseModel):
    """
    LLM 情感分析的结构化输出模型。
    
    为什么用 Pydantic？
    - 自动验证输入类型
    - 提供清晰的 Schema（可导出 JSON Schema 给 LLM）
    - model_dump() 方便序列化
    """
    sentiment: SentimentEnum = Field(
        ...,  # ... 表示必填
        description="情感分类结果"
    )
    confidence: float = Field(
        ...,
        ge=0.0,   # greater than or equal to
        le=1.0,   # less than or equal to
        description="置信度 0-1"
    )
    reasoning: str = Field(
        default="",  # 可选字段，默认空字符串
        max_length=500,
        description="分析理由"
    )
    keywords: List[str] = Field(
        default_factory=list,  # 默认空列表
        description="关键情感词"
    )

# 从 LLM 返回的 JSON 创建模型
raw_llm_output = {
    "sentiment": "positive",
    "confidence": 0.92,
    "reasoning": "用户表达了满意和喜悦的情绪",
    "keywords": ["满意", "开心", "推荐"]
}

result = SentimentResult.model_validate(raw_llm_output)
print("✅ 验证通过！")
print(f"  情感: {result.sentiment.value}")
print(f"  置信度: {result.confidence:.0%}")
print(f"  关键词: {result.keywords}")
print()

# 导出为字典（序列化）
print("model_dump():")
print(f"  {result.model_dump()}")
print()

# 导出 JSON Schema（给 LLM 的 function calling 用）
print("JSON Schema (前 200 字符):")
schema_json = result.model_json_schema()
print(f"  {str(schema_json)[:200]}...")

In [ ]:
# ============================================================
# 示例 2: 字段验证器（field_validator）
# ============================================================

class EmailAnalysis(BaseModel):
    """
    邮件分析结果模型 —— 展示高级验证。
    
    field_validator 用于：
    - 清洗输入（条纹空格、统一格式）
    - 业务规则验证（置信度合理性检查）
    - 跨字段验证（model_validator）
    """
    subject: str = Field(..., min_length=1, max_length=200)
    category: Literal["spam", "important", "normal", "promotional"] = Field(...)
    confidence: float = Field(..., ge=0.0, le=1.0)
    processed_at: Optional[datetime] = None
    
    @field_validator("subject")
    @classmethod
    def strip_subject(cls, v: str) -> str:
        """
        清洗 subject 字段：去首尾空格。
        
        何时使用：
        - LLM 经常在输出前后加多余空格
        - 自动纠正常见的格式问题
        """
        cleaned = v.strip()
        if not cleaned:
            raise ValueError("主题不能为空")
        return cleaned
    
    @field_validator("confidence")
    @classmethod
    def round_confidence(cls, v: float) -> float:
        """
        将置信度四舍五入到 4 位小数。
        
        为什么：
        - LLM 可能返回 0.923456789 这样的超长浮点数
        - 标准化输出格式
        """
        return round(v, 4)

# 测试验证器
try:
    # 模拟 LLM 带空格的输出
    raw_output = {
        "subject": "  关于项目的紧急通知  ",
        "category": "important",
        "confidence": 0.87654321,
    }
    
    analysis = EmailAnalysis.model_validate(raw_output)
    print("✅ 验证通过")
    print(f"  subject (清洗后): '{analysis.subject}'")
    print(f"  confidence (四舍五入): {analysis.confidence}")
    print(f"  processed_at (默认): {analysis.processed_at}")
    
except ValidationError as e:
    print(f"❌ 验证失败: {e}")

In [ ]:
# ============================================================
# 示例 3: 嵌套模型 —— 复杂结构化输出
# ============================================================

class Entity(BaseModel):
    """命名实体"""
    name: str = Field(..., description="实体名称")
    entity_type: Literal["person", "organization", "location", "date", "product"] = Field(...)
    mentions: int = Field(default=1, ge=1, description="提及次数")

class DocumentAnalysis(BaseModel):
    """
    文档完整分析结果 —— 嵌套模型示例。
    
    典型场景：
    - 一次 LLM 调用提取多种结构化信息
    - 文档摘要 + 实体提取 + 情感分析 三合一
    """
    summary: str = Field(..., description="文档摘要")
    entities: List[Entity] = Field(
        default_factory=list,
        max_length=50,  # 最多 50 个实体
        description="提取的命名实体"
    )
    overall_sentiment: SentimentResult  # 嵌套另一个 Pydantic 模型！
    word_count: int = Field(..., ge=0, description="原文词数")
    language: Literal["zh", "en", "other"] = Field(default="zh")

# 模拟 LLM 返回的复杂 JSON
llm_output = {
    "summary": "本文讨论了 AI 在医疗领域的应用前景",
    "entities": [
        {"name": "GPT-5", "entity_type": "product", "mentions": 3},
        {"name": "北京协和医院", "entity_type": "organization", "mentions": 1},
        {"name": "张医生", "entity_type": "person", "mentions": 5},
    ],
    "overall_sentiment": {
        "sentiment": "positive",
        "confidence": 0.85,
        "keywords": ["突破", "创新", "希望"]
    },
    "word_count": 1200,
    "language": "zh"
}

doc = DocumentAnalysis.model_validate(llm_output)
print("✅ 嵌套模型验证成功！")
print(f"  摘要: {doc.summary[:50]}...")
print(f"  实体数量: {len(doc.entities)}")
for entity in doc.entities:
    print(f"    - {entity.name} ({entity.entity_type}) x{entity.mentions}")
print(f"  情感: {doc.overall_sentiment.sentiment.value} ({doc.overall_sentiment.confidence:.0%})")
print()

# model_dump 会递归导出所有嵌套模型
print("完整导出 (model_dump):")
import json
print(json.dumps(doc.model_dump(), indent=2, ensure_ascii=False)[:300] + "...")

### ✏️ 练习 3: Pydantic

1. 定义一个 `ToolCall(BaseModel)` 包含 `name`, `arguments` (dict), `id` (str)。
2. 定义一个 `AgentResponse(BaseModel)` 包含 `thought` (str), `tool_calls` (List[ToolCall] 可选), `final_answer` (str 可选)。
3. 添加 `@field_validator` 确保 `tool_calls` 和 `final_answer` 至少有一个不为空。

---
## 4. Dataclasses —— 轻量级数据结构

### 为什么要学？
Dataclass 是标准库中最常用的数据结构定义方式。比命名元组灵活，比 Pydantic 轻量。
适合不需要验证的场景：Agent 状态缓存、工具结果、评估分数等。

In [ ]:
"""
Dataclasses 详解
=================
关键特性：
- @dataclass 自动生成 __init__, __repr__, __eq__
- field(default_factory=list) 正确处理可变默认值
- frozen=True 创建不可变对象（适合缓存键）
- 与 Pydantic 的互补关系
"""
from dataclasses import dataclass, field, asdict, fields
from typing import List, Optional, Dict
from datetime import datetime
import hashlib

print("✅ Dataclasses 可用（标准库，无需安装）")

# ============================================================
# 示例 1: 基本 Dataclass —— 工具调用结果
# ============================================================

@dataclass
class ToolResult:
    """
    工具调用结果的数据类。
    
    为什么用 dataclass 而不是 Pydantic？
    - 不需要验证（工具本身已保证格式正确）
    - 零依赖，性能更好
    - frozen=True 确保结果不可变
    """
    tool_name: str
    success: bool
    output: str
    latency_ms: float
    
    # field(default_factory=...) 是可变默认值的正确写法
    metadata: Dict[str, str] = field(default_factory=dict)
    
    # field(default_factory=datetime.now) 记录创建时间
    created_at: datetime = field(default_factory=datetime.now)

# 创建实例
result = ToolResult(
    tool_name="web_search",
    success=True,
    output="共找到 42 条相关结果",
    latency_ms=234.5,
    metadata={"query": "LangGraph 教程", "source": "bing"}
)

print(f"工具: {result.tool_name}")
print(f"耗时: {result.latency_ms}ms")
print(f"自动生成的 __repr__: {result!r}"[:120] + "...")
print(f"转为字典: {asdict(result)}")

In [ ]:
# ============================================================
# 示例 2: frozen=True —— 不可变数据类
# ============================================================

@dataclass(frozen=True)
class EvaluationScore:
    """
    评估分数 —— 不可变数据类。
    
    为什么 frozen=True？
    - 评分创建后不应被修改（数据完整性）
    - 可安全用作 dict key 和 set 元素
    - 线程安全
    
    __post_init__ 不能直接修改字段，需要用 object.__setattr__
    """
    evaluator_name: str
    score: float
    max_score: float = 10.0
    
    def __post_init__(self):
        """
        初始化后验证。
        frozen=True 中只能用 object.__setattr__ 修改字段。
        """
        if not (0 <= self.score <= self.max_score):
            raise ValueError(f"分数 {self.score} 不在 [0, {self.max_score}] 范围内")
    
    @property
    def normalized(self) -> float:
        """归一化到 0-1 的分数"""
        return self.score / self.max_score

score = EvaluationScore(evaluator_name="GPT-4o Judge", score=8.5)
print(f"评分者: {score.evaluator_name}")
print(f"分数: {score.score}/{score.max_score} ({score.normalized:.0%})")

# frozen=True → 修改会报错
try:
    score.score = 9.0  # dataclasses.FrozenInstanceError
except Exception as e:
    print(f"❌ 修改被阻止: {type(e).__name__}")

# 但可以创建新的（不可变数据的标准做法）
new_score = EvaluationScore(evaluator_name="GPT-4o Judge", score=9.0)
print(f"✅ 新分数: {new_score.score}/{new_score.max_score}")

In [ ]:
# ============================================================
# 示例 3: Dataclass vs Pydantic 对比总结
# ============================================================

from pydantic import BaseModel as PydanticModel

# --- 用两种方式定义相同的模型 ---

@dataclass
class AgentStateDataclass:
    messages: List[str] = field(default_factory=list)
    temperature: float = 0.7
    
    def add_message(self, msg: str) -> None:
        self.messages.append(msg)

class AgentStatePydantic(PydanticModel):
    messages: List[str] = Field(default_factory=list)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    
    def add_message(self, msg: str) -> None:
        self.messages.append(msg)

print("=" * 60)
print("Dataclass vs Pydantic 对比")
print("=" * 60)

# Dataclass: 无验证，接受非法值
dc = AgentStateDataclass(temperature=999.0)
print(f"\nDataclass (无验证): temperature={dc.temperature}  ← 接受非法值")

# Pydantic: 有验证，拒绝非法值
try:
    pyd = AgentStatePydantic(temperature=999.0)
except ValidationError as e:
    print(f"Pydantic (有验证): 拒绝 temperature=999  ← ValidationError")

# 性能对比
import timeit
dc_time = timeit.timeit(lambda: AgentStateDataclass(), number=10000)
pyd_time = timeit.timeit(lambda: AgentStatePydantic(), number=10000)
print(f"\n创建 10000 次耗时:")
print(f"  Dataclass: {dc_time:.4f}s  ← 更快（无验证开销）")
print(f"  Pydantic:  {pyd_time:.4f}s  ← 较慢（有验证开销）")

print("""
┌─────────────────┬────────────────────┬──────────────────────┐
│      特性       │     @dataclass      │    Pydantic          │
├─────────────────┼────────────────────┼──────────────────────┤
│ 类型验证        │ 仅静态检查         │ 运行时强制验证       │
│ JSON Schema     │ 需手动             │ model_json_schema()  │
│ 序列化          │ asdict()           │ model_dump()         │
│ 性能            │ 极快（标准库）     │ 较慢（有验证开销）   │
│ 依赖            │ 无（标准库）       │ 需安装 pydantic      │
│ 适用场景        │ 内部状态、缓存     │ 外部输入、API、配置  │
│ 对象可变性      │ 可选 frozen=True   │ 可选 model_config     │
└─────────────────┴────────────────────┴──────────────────────┘
""")

### ✏️ 练习 4: Dataclasses

1. 用 `@dataclass(frozen=True)` 定义一个 `PromptTemplate`，包含 `name`, `template_str`, `variables` (List[str])。
2. 添加 `render(**kwargs)` 方法，用传入参数填充模板字符串。
3. 用 `__post_init__` 验证 `template_str` 中所有 `{变量}` 都在 `variables` 列表中。

---
## 5. Type Hints 最佳实践

### 为什么要学？
Type Hints 是 Python 现代化开发的基石。在 AI 项目中，函数签名复杂、数据嵌套深，
没有类型提示的代码很快就会变成维护噩梦。良好的类型提示 = 自文档化 + IDE 智能补全 + mypy 静态检查。

In [ ]:
"""
Type Hints 最佳实践 —— AI 应用常用模式
========================================
覆盖：Optional, Union, List[Dict], Callable, Protocol, TypeVar
"""
from typing import (
    Optional, Union, List, Dict, Any, 
    Callable, Protocol, TypeVar, Literal,
    Sequence, Mapping, Iterator, overload
)
from collections.abc import Sequence as Seq, Mapping as Map

print("✅ 类型提示导入成功")

# ============================================================
# 模式 1: Optional —— 可为 None 的值
# ============================================================

# Optional[X] 等价于 Union[X, None]
def get_api_key(provider: str, fallback: Optional[str] = None) -> Optional[str]:
    """
    获取 API Key。
    
    返回 Optional[str] 表示可能返回 None。
    调用方必须处理 None 的情况。
    """
    import os
    key = os.getenv(f"{provider.upper()}_API_KEY")
    return key if key else fallback

# 调用方正确处理 Optional
key = get_api_key("openai", fallback="sk-test-key")
if key is not None:
    print(f"API Key 已获取: {key[:10]}...")  # 类型收窄后安全

print()

# ============================================================
# 模式 2: Callable —— 回调函数类型
# ============================================================

# 定义一个 LLM 调用回调的类型
OnTokenCallback = Callable[[str, int], None]
#                              ^^^  ^^^    ^^^^
#                             token index  return

def stream_llm_response(
    prompt: str,
    on_token: Optional[OnTokenCallback] = None,
    on_complete: Optional[Callable[[], None]] = None,
) -> str:
    """
    模拟流式 LLM 响应，支持回调。
    
    Callable 是 Python 中回调函数的标准类型标注方式。
    """
    response = f"[模拟回复] 关于 '{prompt[:20]}...' 的回答"
    
    if on_token:
        for i in range(3):  # 模拟 3 个 token
            on_token(f"token_{i}", i)
    
    if on_complete:
        on_complete()
    
    return response

# 使用回调
def my_token_handler(token: str, idx: int) -> None:
    print(f"  → Token {idx}: {token}")

result = stream_llm_response(
    prompt="解释量子计算",
    on_token=my_token_handler,
    on_complete=lambda: print("  ✅ 回复完成"),
)
print(f"完整回复: {result}")

In [ ]:
# ============================================================
# 模式 3: Protocol —— 结构化子类型（鸭子类型）
# ============================================================

class LLMProvider(Protocol):
    """
    LLM 提供者的协议定义。
    
    为什么用 Protocol？
    - 不需要继承同一个基类
    - 只要实现了相同的方法签名，就是合法的 LLMProvider
    - 适合定义“鸭子类型”接口
    
    在课程中：统一的 LLM 接口就是通过 Protocol 定义的。
    """
    async def agenerate(self, prompt: str) -> str:
        """异步生成回复"""
        ...
    
    def count_tokens(self, text: str) -> int:
        """计算 token 数量"""
        ...

# 任何实现了这两个方法的类都是 LLMProvider
class MockLLM:
    """即使不继承 LLMProvider，只要方法匹配，mypy 就认为它是 LLMProvider"""
    async def agenerate(self, prompt: str) -> str:
        return f"Mock 回复: {prompt[:30]}..."
    
    def count_tokens(self, text: str) -> int:
        return len(text) // 4  # 简单估算

def use_provider(provider: LLMProvider) -> None:
    """接受任何实现 LLMProvider 协议的对象"""
    print(f"Token 数: {provider.count_tokens('Hello World')}")

use_provider(MockLLM())  # ✅ 通过
print("✅ Protocol 使得 MockLLM 可以作为 LLMProvider 使用")
print()

# ============================================================
# 模式 4: 复杂嵌套类型 —— AI 项目中最常见
# ============================================================

# 定义清晰的类型别名
MessageDict = Dict[str, Any]
ChatHistory = List[MessageDict]
SearchResults = List[Dict[str, Union[str, float]]]

def process_chat(
    history: ChatHistory,
    new_message: str,
) -> tuple[ChatHistory, Optional[SearchResults]]:
    """
    处理聊天消息。
    
    使用类型别名让函数签名更清晰。
    """
    history.append({"role": "user", "content": new_message})
    history.append({"role": "assistant", "content": f"回复: {new_message[:20]}..."})
    return history, None

# 使用
chat_history: ChatHistory = [
    {"role": "system", "content": "你是一个有帮助的助手"}
]
new_history, results = process_chat(chat_history, "今天天气怎么样？")
print(f"对话轮数: {len(new_history)}")
print(f"搜索结果: {results}")

print()

# ============================================================
# 模式 5: TypeVar —— 泛型函数
# ============================================================

T = TypeVar("T")

def ensure_list(value: Union[T, List[T], None]) -> List[T]:
    """
    统一处理单值和列表的输入。
    
    常见场景：API 参数可能传 "gpt-4" 或 ["gpt-4", "gpt-4o"]，
    内部统一转为列表处理。
    """
    if value is None:
        return []
    if not isinstance(value, list):
        return [value]
    return value

print(f"ensure_list('hello'): {ensure_list('hello')}")
print(f"ensure_list(['a', 'b']): {ensure_list(['a', 'b'])}")
print(f"ensure_list(None): {ensure_list(None)}")

print()
print("💡 类型检查技巧:")
print("  - 运行 mypy: mypy your_file.py")
print("  - VS Code: 安装 Pylance 扩展")
print("  - PyCharm: 内置类型检查器")

### ✏️ 练习 5: Type Hints

1. 定义类型别名 `Embedding = List[float]` 和 `DocumentID = str`。
2. 编写函数 `def cosine_similarity(a: Embedding, b: Embedding) -> float`。
3. 编写函数 `def search(query_embedding: Embedding, index: Dict[DocumentID, Embedding], top_k: int = 5) -> List[DocumentID]`。
4. 运行 mypy 检查类型。

---
## 6. 上下文管理器（Context Managers）

### 为什么要学？
上下文管理器是 Python 资源管理的标准范式。在 AI 项目中：
- 数据库连接 / 向量数据库 session
- 文件处理（PDF、JSONL）
- HTTP session（aiohttp.ClientSession）
- 计时器、临时目录清理

所有需要"获取 → 使用 → 释放"模式的资源都应该用上下文管理器。

In [ ]:
"""
上下文管理器：同步 + 异步
==========================
三种实现方式：
1. 类实现 __enter__ / __exit__
2. contextlib.contextmanager 装饰器（yield 模式）
3. 异步上下文管理器 __aenter__ / __aexit__
"""
from contextlib import contextmanager, asynccontextmanager
import asyncio
import time

print("✅ contextlib 导入成功")

# ============================================================
# 方式 1: 类实现 —— 适用于复杂状态管理
# ============================================================

class Timer:
    """
    计时器上下文管理器。
    
    用途：测量代码块执行时间（LLM 调用延迟、检索耗时等）
    
    何时使用类实现：
    - 需要在 __enter__ 中做复杂初始化
    - 需要在 __exit__ 中根据异常类型做不同处理
    - 对象本身需要暴露额外方法
    """
    def __init__(self, description: str = "操作"):
        self.description = description
        self.elapsed: float = 0.0
    
    def __enter__(self):
        """进入上下文：记录开始时间"""
        self.start = time.perf_counter()
        print(f"⏱️  开始: {self.description}...")
        return self  # 返回 self 给 as 变量
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        退出上下文：计算耗时。
        
        参数说明：
        - exc_type: 异常类型（无异常时为 None）
        - exc_val: 异常值
        - exc_tb: traceback
        
        返回 True 会吞掉异常（通常不这样做）
        """
        self.elapsed = time.perf_counter() - self.start
        status = "❌ 失败" if exc_type else "✅ 完成"
        print(f"⏱️  {status}: {self.description} (耗时 {self.elapsed:.3f}s)")
        return False  # 不吞异常

# 使用类实现的上下文管理器
with Timer("模拟 LLM 调用") as timer:
    time.sleep(0.5)  # 模拟 API 调用
    print("  → 正在调用 GPT-4o...")

print(f"访问耗时属性: {timer.elapsed:.3f}s")
print()

In [ ]:
# ============================================================
# 方式 2: contextmanager 装饰器 —— 适用于简单场景
# ============================================================

@contextmanager
def managed_file(path: str, mode: str = "r"):
    """
    带异常处理的文件上下文管理器。
    
    为什么用 @contextmanager：
    - 代码更简洁，yield 前是 __enter__, yield 后是 __exit__
    - try/finally 保证资源释放
    
    典型场景：
    - 打开/关闭文件
    - 数据库事务的开始/提交/回滚
    - 临时修改环境变量后恢复
    """
    print(f"📂 打开文件: {path}")
    try:
        # Python 的 open() 本身就是上下文管理器，这里演示嵌套
        with open(path, mode, encoding="utf-8") as f:
            yield f  # ← 这里暂停，把控制权交给 with 块中的代码
    except FileNotFoundError:
        print(f"  文件不存在: {path}")
        raise
    finally:
        print(f"📂 关闭文件: {path}")

@contextmanager
def temp_env_var(key: str, value: str):
    """
    临时设置环境变量，退出时自动恢复。
    
    AI 项目常用场景：
    - 测试时临时切换 API endpoint
    - 临时启用/禁用代理
    """
    import os
    old_value = os.environ.get(key)
    os.environ[key] = value
    print(f"🔧 设置 {key}={value}")
    try:
        yield
    finally:
        if old_value is None:
            del os.environ[key]
        else:
            os.environ[key] = old_value
        print(f"🔧 恢复 {key}={old_value}")

# 演示：创建临时文件并读写
import tempfile, os
tmp_path = os.path.join(tempfile.gettempdir(), "test_context.txt")

with managed_file(tmp_path, "w") as f:
    f.write("Hello from context manager!")
    print(f"  → 写入成功")

# 清理
os.remove(tmp_path)
print()

In [ ]:
# ============================================================
# 方式 3: 异步上下文管理器 —— async with
# ============================================================

class AsyncAPISession:
    """
    异步 API Session 管理器。
    
    模拟 aiohttp.ClientSession 的行为。
    实际项目中：
    - async with aiohttp.ClientSession() as session:
    - async with asyncpg.create_pool() as pool:
    """
    def __init__(self, api_name: str):
        self.api_name = api_name
        self.connected = False
    
    async def __aenter__(self):
        """异步进入：建立连接"""
        print(f"🔌 连接 {self.api_name}...")
        await asyncio.sleep(0.3)  # 模拟连接建立
        self.connected = True
        print(f"🔌 {self.api_name} 已连接")
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        """异步退出：关闭连接"""
        print(f"🔌 断开 {self.api_name}...")
        await asyncio.sleep(0.1)  # 模拟优雅关闭
        self.connected = False
        print(f"🔌 {self.api_name} 已断开")
        return False
    
    async def call(self, prompt: str) -> str:
        """模拟 API 调用"""
        if not self.connected:
            raise RuntimeError("Session 未连接！")
        await asyncio.sleep(0.2)
        return f"[{self.api_name}] 回复: {prompt[:30]}..."

@asynccontextmanager
async def managed_async_session(api_name: str):
    """
    使用 @asynccontextmanager 的异步版本。
    
    比类实现更简洁，适合简单场景。
    """
    session = AsyncAPISession(api_name)
    try:
        await session.__aenter__()
        yield session
    finally:
        await session.__aexit__(None, None, None)

async def demo_async_context():
    """演示异步上下文管理器的完整用法"""
    print("=== 异步上下文管理器演示 ===\n")
    
    # 使用类实现的异步上下文管理器
    async with AsyncAPISession("OpenAI") as session:
        result = await session.call("什么是 LangGraph？")
        print(f"  📨 {result}")
    
    print()
    
    # 使用装饰器实现的异步上下文管理器
    async with managed_async_session("Anthropic") as session:
        result = await session.call("什么是 Vector Store？")
        print(f"  📨 {result}")
    
    print("\n✅ 异步上下文管理器演示完成")

asyncio.run(demo_async_context())

### ✏️ 练习 6: Context Managers

1. 用类实现一个 `DatabaseTransaction` 上下文管理器，`__enter__` 输出 "BEGIN"，`__exit__` 时如果无异常输出 "COMMIT"，有异常输出 "ROLLBACK"。
2. 用 `@contextmanager` 实现 `change_directory(path)`，临时切换到指定目录，退出时恢复。
3. 用 `@asynccontextmanager` 实现 `rate_limited_api(api_name, max_rps)`，限速调用 API。

---
## 7. 综合实战：文档处理 Pipeline

将以上所有概念整合为一个完整的异步文档处理函数：
- TypedDict 定义配置
- Pydantic 定义输出结构
- Dataclass 定义中间状态
- async/await 并发处理
- 上下文管理器管理资源
- 完整的类型提示

In [ ]:
"""
综合实战：异步文档处理 Pipeline
=================================
场景：接收一个文件路径，调用多个 LLM API 进行分析，返回结构化结果。

这个例子展示了 Phase 01 所有核心概念的协同使用。
"""
import asyncio
import time
from dataclasses import dataclass, field
from typing import TypedDict, Optional, List, Dict, Any, Literal
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager

# ============================================================
# 1. TypedDict: 定义处理配置
# ============================================================
class ProcessConfig(TypedDict, total=False):
    """
    文档处理配置。
    total=False 表示所有字段都是可选的。
    """
    model: str                              # LLM 模型名称
    temperature: float                      # 温度参数
    max_tokens: int                         # 最大 token 数
    enable_summary: bool                    # 是否生成摘要
    enable_entities: bool                   # 是否提取实体
    enable_sentiment: bool                  # 是否分析情感
    extract_keywords: Optional[List[str]]   # 额外关键词提取


# ============================================================
# 2. Pydantic: 定义输出结构
# ============================================================
class EntityOutput(BaseModel):
    """实体提取输出（Pydantic 验证）"""
    name: str
    entity_type: Literal["person", "org", "location", "product", "other"]
    importance: float = Field(default=0.5, ge=0.0, le=1.0)

class ProcessedDocument(BaseModel):
    """
    文档处理完整输出。
    
    这是整个 pipeline 的最终产物。
    所有字段都有 Pydantic 验证保证数据质量。
    """
    file_path: str
    summary: Optional[str] = None           # 摘要（可选）
    entities: List[EntityOutput] = Field(default_factory=list)
    sentiment: Optional[Literal["positive", "negative", "neutral"]] = None
    word_count: int = Field(default=0, ge=0)
    processing_time_ms: float = Field(default=0.0, ge=0.0)
    errors: List[str] = Field(default_factory=list)
    
    @property
    def success(self) -> bool:
        """快速判断处理是否完全成功"""
        return len(self.errors) == 0


# ============================================================
# 3. Dataclass: 中间处理状态
# ============================================================
@dataclass
class ProcessingMetrics:
    """
    处理过程中的度量数据。
    
    用 dataclass 而非 Pydantic：
    - 内部状态，不需要序列化验证
    - 性能敏感（高频更新）
    - 可变状态
    """
    total_calls: int = 0
    failed_calls: int = 0
    tokens_used: int = 0
    estimated_cost: float = 0.0
    
    def record_success(self, tokens: int, cost: float) -> None:
        """记录一次成功的 API 调用"""
        self.total_calls += 1
        self.tokens_used += tokens
        self.estimated_cost += cost
    
    def record_failure(self) -> None:
        """记录一次失败的 API 调用"""
        self.total_calls += 1
        self.failed_calls += 1


# ============================================================
# 4. 上下文管理器: API Session 管理
# ============================================================
@asynccontextmanager
async def api_session(model: str):
    """
    管理 API Session 的异步上下文管理器。
    
    在实际项目中，这里会创建 aiohttp.ClientSession 或
    OpenAI/Anthropic 的 AsyncClient。
    
    退出时自动清理连接，防止资源泄漏。
    """
    print(f"  🔌 创建 API Session: {model}")
    # 模拟: session = AsyncOpenAI(api_key=...)
    yield {"model": model, "connected": True}
    print(f"  🔌 关闭 API Session: {model}")
    # 模拟: await session.close()


# ============================================================
# 5. 主函数: 将所有概念组合在一起
# ============================================================

async def process_document(
    file_path: str,
    config: ProcessConfig,
) -> ProcessedDocument:
    """
    异步处理单个文档 —— Phase 01 概念的综合应用。
    
    Args:
        file_path: 待处理的文档路径
        config: TypedDict 定义的处理配置
    
    Returns:
        ProcessedDocument: Pydantic 验证的结构化结果
    
    涉及的概念：
    - async/await: 异步 API 调用
    - TypedDict: 配置定义
    - Pydantic: 输出验证
    - Dataclass: 度量收集
    - Context Manager: 资源管理
    - Type Hints: 完整的类型标注
    """
    start = time.perf_counter()
    errors: List[str] = []
    metrics = ProcessingMetrics()  # Dataclass
    
    # 模拟读取文件内容
    print(f"📄 处理文件: {file_path}")
    await asyncio.sleep(0.1)  # 模拟 I/O
    content = f"[文件内容] {file_path} 包含的示例文本内容..."
    word_count = len(content.split())
    
    # 使用上下文管理器管理 API 连接
    async with api_session(config.get("model", "gpt-4o")) as session:
        # 并发执行多个分析任务
        tasks = []
        
        if config.get("enable_summary", True):
            tasks.append(("summary", analyze_summary(content, session)))
        if config.get("enable_entities", True):
            tasks.append(("entities", analyze_entities(content, session)))
        if config.get("enable_sentiment", True):
            tasks.append(("sentiment", analyze_sentiment(content, session)))
        
        # asyncio.gather 并发执行
        results = await asyncio.gather(
            *[task for _, task in tasks],
            return_exceptions=True
        )
        
        # 处理结果
        summary = None
        entities: List[EntityOutput] = []
        sentiment = None
        
        for (task_name, _), result in zip(tasks, results):
            if isinstance(result, Exception):
                errors.append(f"{task_name}: {result}")
                metrics.record_failure()
            else:
                tokens = len(str(result)) // 4  # 简单 token 估算
                cost = tokens * 0.00001  # 模拟每 token 成本
                metrics.record_success(tokens, cost)
                
                if task_name == "summary":
                    summary = result
                elif task_name == "entities":
                    entities = result
                elif task_name == "sentiment":
                    sentiment = result
    
    elapsed_ms = (time.perf_counter() - start) * 1000
    
    # 构建最终结果（Pydantic 验证）
    return ProcessedDocument(
        file_path=file_path,
        summary=summary,
        entities=entities,
        sentiment=sentiment,
        word_count=word_count,
        processing_time_ms=round(elapsed_ms, 2),
        errors=errors,
    )


# ============================================================
# 模拟分析函数（实际项目中调用 LLM API）
# ============================================================

async def analyze_summary(content: str, session: dict) -> str:
    """生成文档摘要（异步模拟）"""
    await asyncio.sleep(0.3)
    return f"这是一份关于 {content[:20]} 的摘要..."

async def analyze_entities(content: str, session: dict) -> List[EntityOutput]:
    """提取命名实体（异步模拟）"""
    await asyncio.sleep(0.4)
    return [
        EntityOutput(name="LangGraph", entity_type="product", importance=0.9),
        EntityOutput(name="Anthropic", entity_type="org", importance=0.8),
    ]

async def analyze_sentiment(content: str, session: dict) -> Literal["positive", "negative", "neutral"]:
    """分析情感（异步模拟）"""
    await asyncio.sleep(0.2)
    return "positive"


# ============================================================
# 运行演示
# ============================================================

async def main():
    """主函数：测试综合实战代码"""
    print("=" * 60)
    print("  Phase 01 综合实战：异步文档处理 Pipeline")
    print("=" * 60)
    print()
    
    # TypedDict 配置
    config: ProcessConfig = {
        "model": "gpt-4o",
        "temperature": 0.0,
        "max_tokens": 2048,
        "enable_summary": True,
        "enable_entities": True,
        "enable_sentiment": True,
    }
    
    # 处理文档
    result = await process_document(
        file_path="/data/reports/2024-annual-report.txt",
        config=config,
    )
    
    # 输出结果
    print()
    print("=" * 60)
    print("  处理结果")
    print("=" * 60)
    print(f"  文件: {result.file_path}")
    print(f"  状态: {'✅ 成功' if result.success else '❌ 有错误'}")
    print(f"  词数: {result.word_count}")
    print(f"  摘要: {result.summary}")
    print(f"  情感: {result.sentiment}")
    print(f"  实体: {len(result.entities)} 个")
    for e in result.entities:
        print(f"    - {e.name} ({e.entity_type}) 重要性: {e.importance}")
    print(f"  耗时: {result.processing_time_ms}ms")
    if result.errors:
        print(f"  错误: {result.errors}")
    
    # model_dump 导出完整 JSON
    print()
    print("  完整 JSON 输出:")
    print(f"  {result.model_dump_json(indent=2)[:300]}...")
    
    return result

# 运行
final_result = asyncio.run(main())
print(f"\n🎉 类型检查: {type(final_result).__name__}")  # ProcessedDocument

---
## 📋 学习检查清单

完成本 Notebook 后，你应该能够：

- [ ] 写出包含错误处理的 async/await 代码
- [ ] 用 TypedDict 定义字典结构并获得 IDE 自动补全
- [ ] 用 Pydantic BaseModel 验证 LLM 的结构化输出
- [ ] 用 Dataclass 定义轻量级内部数据结构
- [ ] 区分何时用 TypedDict / Pydantic / Dataclass
- [ ] 写出完整的 Type Hints 并获得 mypy 零错误
- [ ] 用上下文管理器管理文件和网络资源
- [ ] 理解 LangGraph State 背后的 TypedDict 机制
- [ ] 理解 LangChain 结构化输出背后的 Pydantic 机制

**下一步：** 02-llm-api-basics.ipynb —— 使用这些 Python 技能调用真正的 LLM API